# vLLM trace collection — campaign run (one model per session)

Runs `src.capture.vllm_collect` against the canonical corpus and uploads to the **new** vLLM traces
repo `Ryze242005/vllm-traces` (the old llama.cpp traces at `Ryze242005/moe-traces` are left untouched — D3).

**Before running:**
1. GPU **T4 ×2**, **Internet On**.
2. **Add Input → `ryzewtf/moe-corpus`** (the corpus dataset; do NOT let anything re-fetch it — HF
   streaming is unpinned and would draw different documents, breaking T4.3 byte-identity).
3. Kaggle **Secrets**: `HF_TOKEN` (write scope) and `GITHUB_TOKEN`.
4. Set **`MODEL_KEY`** in the collect cell. One model per session.

Run top-to-bottom once; do **not** restart the kernel (Kaggle restart reverts the pip installs).
The ledger is resumable — re-running continues where an interrupted session stopped. Paste back the
`… collected …` summary line.


In [ ]:
# ============================================================================
# Cell 1 — runtime audit (subprocess; does NOT import torch into the kernel).
# ============================================================================
import subprocess
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free,compute_cap",
     "--format=csv"], capture_output=True, text=True).stdout, flush=True)
print("EXPECT two rows, compute_cap 7.5.")


In [ ]:
# ============================================================================
# Cell 2 — install vLLM + transformers (P0's confirmed recipe). Do NOT restart after.
# ============================================================================
VLLM_VERSION = "0.10.2"
import subprocess, sys


def sh(args):
    print("$", " ".join(args), flush=True)
    p = subprocess.run(args, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print((p.stdout or "")[-3000:], flush=True)
    print("exit:", p.returncode, flush=True)
    return p.returncode


sh([sys.executable, "-m", "pip", "uninstall", "-y",
    "vllm", "torch", "torchvision", "torchaudio", "transformers"])
sh([sys.executable, "-m", "pip", "install", "-q",
    f"vllm=={VLLM_VERSION}", "transformers==4.55.2"])
sh([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"])

chk = subprocess.run(
    [sys.executable, "-c", "import torch, vllm; from vllm import LLM; print('IMPORT_OK')"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(chk.stdout, flush=True)
print(">>> Proceed only if IMPORT_OK printed. DO NOT restart the kernel.")


In [ ]:
# ============================================================================
# Cell 3 — clone the repo, put src/ on sys.path AND PYTHONPATH (TP=2 workers need it).
# ============================================================================
import os, subprocess, sys
from pathlib import Path

GIT_URL = "https://github.com/ryzewtf/GenAI-IA-1.git"
GIT_REF = "VLLM_PORT"
REPO = Path("/kaggle/working/repo")


def run(cmd, cwd=None, check=True, quiet=False):
    if not quiet:
        print("$", " ".join(str(c) for c in cmd), flush=True)
    p = subprocess.run([str(c) for c in cmd], cwd=cwd and str(cwd), text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if p.stdout and not quiet:
        print(p.stdout, flush=True)
    if check and p.returncode != 0:
        raise SystemExit(f"FAILED ({p.returncode}): {' '.join(str(c) for c in cmd)}")
    return p


url = GIT_URL
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if tok:
        url = GIT_URL.replace("https://", f"https://{tok}@")
        print("using GITHUB_TOKEN from Kaggle Secrets")
except Exception:
    print("no GITHUB_TOKEN secret; cloning anonymously (fine if the repo is public)")

if REPO.exists():
    run(["git", "fetch", "--all", "--tags"], cwd=REPO)
    run(["git", "checkout", GIT_REF], cwd=REPO)
    run(["git", "pull", "--ff-only"], cwd=REPO, check=False)
else:
    run(["git", "clone", url, str(REPO)])
    run(["git", "checkout", GIT_REF], cwd=REPO)
print("repo at", run(["git", "rev-parse", "HEAD"], cwd=REPO, quiet=True).stdout.strip())

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ["PYTHONPATH"] = str(REPO) + os.pathsep + os.environ.get("PYTHONPATH", "")
os.chdir(REPO)


In [ ]:
# ============================================================================
# Cell 4 — locate the canonical corpus (mounted dataset) and export HF_TOKEN.
# ============================================================================
import os
from pathlib import Path

# The corpus MUST be the mounted Kaggle dataset — never re-fetched (T4.3 byte-identity).
CORPUS = Path("/kaggle/input/moe-corpus/mixed-v1.jsonl")
CORPUS_NAME = "mixed-v1"
assert CORPUS.exists(), (
    f"{CORPUS} not found — Add Input -> ryzewtf/moe-corpus (Internet On won't fetch it for you)")
print("corpus:", CORPUS, f"({CORPUS.stat().st_size/1e6:.1f} MB)")

# HF write token from Kaggle Secrets -> env for HFBackend. NEVER inline the token.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets")
except Exception as e:
    print("WARNING: no HF_TOKEN secret -", e, "(the upload will fail without it)")


In [ ]:
# ============================================================================
# Cell 5 — collect ONE model against the full corpus and upload to the new vLLM repo.
# ============================================================================
# Tier-1 order: olmoe-0125 -> olmoe-0125-instruct -> olmoe-0924 -> qwen3-30b-a3b.
MODEL_KEY = "olmoe-0125"     # <<< set this per session
TRACE_REPO = "Ryze242005/vllm-traces"  # new repo; old llama.cpp traces untouched (D3)

import subprocess, sys

SCRATCH = "/tmp/vscratch"    # ledger + per-shard scratch (deleted after each verified upload)

argv = [
    sys.executable, "-m", "src.capture.vllm_collect",
    "--model", MODEL_KEY, "--corpus", str(CORPUS), "--corpus-name", CORPUS_NAME,
    "--backend", "hf", "--repo-id", TRACE_REPO, "--scratch", SCRATCH,
    # To resume an interrupted model, add:  "--shards", "10-20"
]
print("$", " ".join(argv), flush=True)
rc = subprocess.run(argv, text=True).returncode
print(f"\n== collector exit {rc}", flush=True)
print(f"{MODEL_KEY}:", "OK — shards on HF" if rc == 0 else "FAILED (see above)")
